In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
from ase.io import read
from collections import Counter
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.core.structure import Structure
from pymatgen.symmetry.groups import SpaceGroup

from evaluations.aggregate_metrics import aggregate

# Common color palette
COLORS = mpl.rcParams['axes.prop_cycle'].by_key()['color']
PADDING = 15

In [5]:
# Configuration - centralized paths and runs
BASE_PATH_GENERATED_STRUCTURES = Path("generated_structures")
BASE_PATH_PREDICTIONS = Path("predictive_models/topological_mp_20/inference/predictions")

# Standard runs for all analyses
RUNS = [
    "mattergen_base",
    "non_topological_dgf2", 
    "topological_dgf2",
    "topological_dgf5",
]

In [7]:
def load_combined_data(runs=None, base_path_generated_structures=None, base_path_predictions=None):
    """Load and combine data for all runs."""
    # Use defaults if not provided
    runs = runs or RUNS
    base_path_generated_structures = base_path_generated_structures or BASE_PATH_GENERATED_STRUCTURES
    base_path_predictions = base_path_predictions or BASE_PATH_PREDICTIONS
    
    rows = []

    for run in runs:
        folder = Path(base_path_generated_structures) / run
        predictions_path = Path(base_path_predictions) / f"{run}/predictions.csv"

        # Check if all required files exist
        labeled_data_matches_path = folder / "labeled_data_matches.csv"
        metrics_path = folder / "metrics.csv"

        if not labeled_data_matches_path.exists():
            from evaluations.labeled_data_matches import create_data_matches_for_dir
            create_data_matches_for_dir(folder)

        if not metrics_path.exists():
            print(f"Metrics file not found in {folder}, skipping.")
            continue

        # Load data
        labeled_data_matches = pd.read_csv(labeled_data_matches_path)
        refinement = pd.read_csv(metrics_path)
        predictions = pd.read_csv(predictions_path)

        # Combine data
        combined = pd.concat(
            [labeled_data_matches, refinement, predictions], axis=1
        )
        combined = combined[:20_000]

        # Calculate metrics
        unique_structures = combined[combined["unique"]]
        unique_non_train = unique_structures[
            (unique_structures["origin_mp_20"] != "train")
            & (unique_structures["origin_mp_20"] != "val")
        ]

        num_unique_topological = (unique_non_train["topological"] == 1).sum()
        num_unique_non_topological = (unique_non_train["topological"] == 0).sum()
        total_unique = num_unique_topological + num_unique_non_topological
        frac_topological = num_unique_topological / total_unique if total_unique > 0 else 0

        # Positive predictions
        unique_non_train_filtered = unique_non_train[unique_non_train["predicted_label"] == 1]
        num_topo_filtered = (unique_non_train_filtered["topological"] == 1).sum()
        num_non_topo_filtered = (unique_non_train_filtered["topological"] == 0).sum()
        total_filtered = num_topo_filtered + num_non_topo_filtered
        frac_topo_filtered = num_topo_filtered / total_filtered if total_filtered > 0 else 0

        rows.append(
            {
                "folder": folder.name,
                "num_generated": len(combined),
                "num_unique": len(unique_structures),
                "num_unique_topological": num_unique_topological,
                "frac_topological": frac_topological,
                "num_unique_topological_filtered": num_topo_filtered,
                "frac_topological_filtered": frac_topo_filtered,
            }
        )

    return pd.DataFrame(rows)


# Load data using defaults
metrics_df = load_combined_data()
metrics_df

,folder,num_generated,num_unique,num_unique_topological,frac_topological,num_unique_topological_filtered,frac_topological_filtered
0,mattergen_base,20000,19457,18,0.375000,17,0.944444
1,non_topological_dgf2,20000,18482,21,0.155556,15,0.937500
2,topological_dgf2,20000,16249,66,0.709677,61,0.924242
3,topological_dgf5,20000,10846,37,0.880952,36,0.923077


In [ ]:
def plot_metric_bar(
    data_dict, metric_name, use_percentage=False, title="", ylabel="", figsize=(6, 6)
):
    """Unified bar chart plotting function."""
    model_names = []
    values = []

    # Handle both DataFrame and dict inputs
    if isinstance(data_dict, pd.DataFrame):
        # Convert DataFrame to dict format
        data_dict = {idx: row for idx, row in data_dict.iterrows()}

    for model, data in data_dict.items():
        if isinstance(data, pd.Series):
            df = data.to_frame().T
        elif isinstance(data, dict):
            df = pd.DataFrame([data])
        else:
            df = data

        if metric_name not in df.columns:
            continue

        if use_percentage:
            value = df[metric_name].iloc[0] * 100 if len(df) == 1 else df[metric_name].mean() * 100
        else:
            value = (
                df[metric_name].iloc[0]
                if len(df) == 1
                else (
                    df[metric_name].sum()
                    if df[metric_name].dtype == bool
                    else df[metric_name].mean()
                )
            )

        model_names.append(str(model))
        values.append(value)

    if not values:
        print(f"No data found for metric: {metric_name}")
        return

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(
        model_names,
        values,
        alpha=0.8,
        color=COLORS[: len(model_names)],
        edgecolor="black",
        linewidth=0.5,
    )

    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            height + max(values) * 0.02,
            f'{value:.1f}{"%" if use_percentage else ""}',
            ha="center",
            va="bottom",
            fontsize=12,
        )

    ax.set_xlabel("Generated Dataset", fontsize=14, labelpad=PADDING)
    ax.set_ylabel(ylabel or metric_name, fontsize=14, labelpad=PADDING)
    if title:
        ax.set_title(title, fontsize=16, pad=PADDING)

    ax.tick_params(axis="x", rotation=45, labelsize=12)
    ax.tick_params(axis="y", labelsize=12)
    ax.set_ylim(0, max(values) * 1.15)

    for label in ax.get_xticklabels():
        label.set_ha("right")

    plt.tight_layout()
    plt.show()


# Convert to dict format for plotting
metrics_dict = {row["folder"]: row for _, row in metrics_df.iterrows()}

# Plot 1: Fraction topological
plot_metric_bar(
    metrics_dict,
    "frac_topological",
    use_percentage=True,
    title="Fraction topological among rediscovered structures",
    ylabel="Fraction Topological (%)",
)

# Plot 2: Absolute numbers
plot_metric_bar(
    metrics_dict,
    "num_unique_topological",
    use_percentage=False,
    title="Absolute number of rediscovered topological structures",
    ylabel="Count",
)

In [ ]:
def load_model_data(runs=None, base_path=None):
    """Load metrics and refinement data for all models."""
    # Use defaults if not provided
    runs = runs or RUNS
    base_path = base_path or BASE_PATH_GENERATED_STRUCTURES
    
    metrics_dfs = {}
    filtered_dfs = {}

    for run in runs:
        metrics_path = base_path / run / "metrics.csv"
        if not metrics_path.exists():
            print(f"Warning: Missing metrics file for {run}")
            continue
            
        metrics_df = pd.read_csv(metrics_path)
        metrics_df["stable_unique"] = metrics_df["stable"] & metrics_df["unique"]
        metrics_dfs[run] = metrics_df
        filtered_dfs[run] = metrics_df[metrics_df["unique"]]

    return metrics_dfs, filtered_dfs


# Load and process data using defaults
metrics_dfs, filtered_dfs = load_model_data()

# Create plots using unified function
plot_metric_bar(metrics_dfs, "unique", use_percentage=True, title="Uniqueness", ylabel="Unique (%)")
plot_metric_bar(
    filtered_dfs,
    "structure_comp_validity",
    use_percentage=True,
    title="Validity",
    ylabel="Valid (%)",
)
plot_metric_bar(filtered_dfs, "stable", use_percentage=True, title="Stability", ylabel="Stable (%)")

In [ ]:
import sys
sys.path.append("./evaluations")

# Run the complete analysis using cached data
def load_data(
    runs=None, base_path_generated_structures=None, base_path_predictions=None, force_reload=False
):
    # Use defaults if not provided
    runs = runs or RUNS
    base_path_generated_structures = base_path_generated_structures or BASE_PATH_GENERATED_STRUCTURES
    base_path_predictions = base_path_predictions or BASE_PATH_PREDICTIONS

    print("Loading aggregated results...")
    results = {}
    for run in runs:
        try:
            results[run] = aggregate(
                Path(base_path_generated_structures) / run, Path(base_path_predictions) / f"{run}.csv"
            )
        except Exception as e:
            print(f"Error loading {run}: {e}")

    print(f"Available datasets: {list(results.keys())}")

    return results


results = load_data()

In [ ]:
def count_elements_in_structures(structures, unique=True):
    """Count element occurrences in a list of structures."""
    all_atomic_numbers = []
    for struct in structures:
        atomic_numbers = [s.Z for s in struct.species]
        if unique:
            atomic_numbers = list(set(atomic_numbers))
        all_atomic_numbers.extend(atomic_numbers)
    return Counter(all_atomic_numbers)


def create_ptable_heatmap(structures, title_suffix="", unique=True):
    import pymatviz as pmv  # install with "uv pip install pymatviz", might require "plotly==6.1.2"

    """Generate and display periodic table heatmap for element distribution."""
    all_counts = count_elements_in_structures(structures, unique=unique)

    fig = pmv.ptable_heatmap_plotly(
        all_counts,
        log=True,
        fmt=".0f",
        colorbar={"title": "Number of structures containing element"},
        cscale_range=(0, None),
    )
    if title_suffix:
        fig.update_layout(title=f"Element Distribution - {title_suffix}")
    fig.show()


def create_comparative_heatmap(counts_dict, dataset_a, dataset_b):
    import pymatviz as pmv  # install with "uv pip install pymatviz", might require "plotly==6.1.2"

    """Create comparative heatmap between two datasets."""
    diff_counts = counts_dict[dataset_a].copy()
    diff_counts.subtract(counts_dict[dataset_b])

    if not diff_counts:
        print(f"No data to compare between {dataset_a} and {dataset_b}")
        return

    largest_count = round(max(abs(min(diff_counts.values())), max(diff_counts.values())), ndigits=2)

    fig = pmv.ptable_heatmap_plotly(
        diff_counts,
        log=False,
        colorscale=["cornflowerblue", "white", "salmon"],
        colorbar={
            "title": "Difference of occurrences",
            "tickvals": [-largest_count, 0, largest_count],
            "ticktext": [f"{-largest_count:.1%}", "0.0%", f"{largest_count:.1%}"],
        },
        cscale_range=(-largest_count, largest_count),
        nan_color="lightgrey",
        fmt=".1%",
    )
    fig.update_layout(title=f"Element Distribution Comparison: {dataset_a} vs {dataset_b}")
    fig.show()


def load_reference_data():
    """Load MP20 reference data."""
    dataset_path = "datasets/mp_20/test.csv"
    dataset = pd.read_csv(dataset_path)
    labels = dataset["topological"].tolist()

    structures_mp20 = [Structure.from_str(s, fmt="cif") for s in dataset["cif"]]
    structures_mp20_test_topological = [s for s, l in zip(structures_mp20, labels) if l == 1]
    structures = {
        "mp20_test": structures_mp20,
        "mp20_test_topological": structures_mp20_test_topological,
    }

    space_groups_mp20 = dataset["spacegroup_number"].tolist()
    space_groups_mp20_test_topological = [sg for sg, l in zip(space_groups_mp20, labels) if l == 1]
    space_groups = {
        "mp20_test": space_groups_mp20,
        "mp20_test_topological": space_groups_mp20_test_topological,
    }

    return structures, space_groups


def plot_space_group_distribution(space_group_datasets, datasets_to_plot, top_n=5):
    """Plot space group distribution for top N groups."""
    space_group_counts = {
        k: Counter(v) for k, v in space_group_datasets.items() if k in datasets_to_plot
    }

    # Create percentage DataFrame
    df = pd.DataFrame.from_dict(space_group_counts).fillna(0)
    df = df[datasets_to_plot]
    df = df.div(df.sum())  # Normalize to percentages

    # Convert space group numbers to symbols
    df.index = [SpaceGroup.from_int_number(int(sg)).symbol for sg in df.index]

    # Get top N space groups
    top_n_values = df.sort_values(by=datasets_to_plot[0], ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(10, 6))

    bars = top_n_values.plot(
        kind="bar",
        ax=ax,
        width=0.75,
        color=["gray"] + COLORS[: len(datasets_to_plot) - 1],
        alpha=0.8,
        edgecolor="black",
        linewidth=0.5,
    )

    # Format y-axis as percentages
    ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1, decimals=1))
    ax.set_ylabel("Percentage (%)", fontsize=14, labelpad=PADDING)
    ax.set_xlabel("Space Group", fontsize=14, labelpad=PADDING)
    ax.set_title(f"Space Group Distribution - Top {top_n} ", fontsize=16, pad=PADDING)

    # Use technical terms in legend
    ax.legend(datasets_to_plot, fontsize=12, frameon=True)
    ax.tick_params(axis="x", rotation=45, labelsize=12)
    ax.tick_params(axis="y", labelsize=12)

    for label in ax.get_xticklabels():
        label.set_ha("right")

    plt.tight_layout()
    plt.show()

def run_structure_analysis(results):
    """Run comprehensive structure analysis using default or custom paths."""
    # Use defaults if not provided

    # Load structure data
    structure_datasets = {
        key: results[key]["structure"] for key in results 
    }
    space_group_datasets = {
        key: results[key]["space_group_number_01"] for key in results
    }

    # Add reference data
    ref_structures, ref_space_groups = load_reference_data()
    structure_datasets.update(ref_structures)
    space_group_datasets.update(ref_space_groups)

    # Run analysis
    print("Running periodic table analysis...")
    for dataset_key in [
        "mattergen_base",
        "mp20_test", 
        "topological_dgf2",
        "mp20_test_topological",
    ]:
        if dataset_key in structure_datasets:
            create_ptable_heatmap(structure_datasets[dataset_key], title_suffix=dataset_key)

    print("Running comparative analysis...")
    if "topological_dgf2" in structure_datasets and "mattergen_base" in structure_datasets:
        # Create element occurrence counts for comparison
        counts_dict = {}
        for key, structures in structure_datasets.items():
            if key in ["topological_dgf2", "mattergen_base"]:
                counts = count_elements_in_structures(structures, unique=True)
                counts_dict[key] = Counter({el: v / len(structures) for el, v in counts.items()})

        create_comparative_heatmap(counts_dict, "topological_dgf2", "mattergen_base")

    print("Running space group analysis...")
    datasets_to_plot = [
        "mp20_test_topological",
        "mattergen_base",
        "non_topological_dgf2",
        "topological_dgf2", 
        "topological_dgf5",
    ]
    available_datasets = [d for d in datasets_to_plot if d in space_group_datasets]
    if available_datasets:
        plot_space_group_distribution(space_group_datasets, available_datasets, top_n=5)


run_structure_analysis(results)